# Calvin Site 1 and SO4/T4 spatial evaluation

Use this notebook from the DeepRootGen checkout. It reads the archived local
predictions by default. The scores use matched core increments and voxel shares.
The [report](../../Calvin_SO4_Parameter_Exploration_2026-11-18/Spatial_Evaluation_2026-11-19/REPORT.md) links the matched tables and seed receipts.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

cwd = Path.cwd()
root = next(p for p in (cwd, *cwd.parents) if (p / "deeprootgen").is_dir())
directory = root / "Calvin_SO4_Parameter_Exploration_2026-11-18/Spatial_Evaluation_2026-11-19"
sys.path.insert(0, str(directory))
import calvin_spatial as spatial

calvin = pd.read_csv(directory / "calvin_spatial_scores.csv")
ranked = pd.read_csv(directory / "calvin_development_ranks.csv")
so4 = pd.read_csv(directory / "so4_spatial_scores.csv")
receipt = spatial.load(directory / "calvin_spatial_receipt.json")
print("Held core groups", ", ".join(receipt["held_groups"]))
print("Training normalizers", receipt["scales"])

## Calvin paired root length density and dry mass

Compare the same cylinder at its measured coordinates and 10 cm depth interval. RLD includes
0.93–2.50 mm roots and mass includes diameters at or above 0.93 mm. Volume-weight increments
within a core and give each core equal weight. Fit on 11 cores and keep the three withheld cores
together through the comparison. Lower numbers indicate closer agreement.

The ranking gives half the weight to paired normalized MAE, one quarter to joint
location–depth total variation and one quarter to depth total variation. Both responses have
equal weight in each term. The two allocation errors use shares on the sampled support.

In [ ]:
use = (
    calvin.panel.isin(["baseline", "combined_28"])
    | (calvin.panel.eq("Calvin-qc") & calvin.candidate.eq("qc_refine35"))
)
columns = ["candidate", "seeds", "rld_nmae", "mass_nmae", "rld_joint_tv", "mass_joint_tv",
           "rld_depth_tv", "mass_depth_tv", "spatial_score"]
for fold in ("Site1_hold", "Site1_hold_qc", "Site5_external"):
    print(fold)
    print(calvin.loc[use & calvin.fold.eq(fold), columns].round(3).to_string(index=False))
    print()

Each line follows a withheld core. Open observation markers indicate scanner-total flags.
The dashed profile uses fitting-core means by depth and predicts the same value at each location.

![Root length density and dry-mass density at withheld core locations](../../Calvin_SO4_Parameter_Exploration_2026-11-18/Spatial_Evaluation_2026-11-19/calvin_held_core_profiles.png)

## Fit using the measured spatial support

The following example repeats scoring for a saved, full-resolution, common-seed panel.
Change `stage` and `candidate` to another complete row in the development table.
The archived prediction table represents mass at unit tissue density. For a new
study, call the same `measure` and `evaluate` functions with prediction rows aligned
to the prepared `target_row` index. Pass `density=1.0` when the mass column already
represents dry mass in grams. Rank with the fitting mask, freeze the input and density,
then evaluate the held locations with reserved simulation seeds. Save the prediction
tables, density and run receipts for the exact seeds used.

In [ ]:
obs, folds, scales = spatial.support()
fit = folds["Site1_fit"]
stage, candidate = "Calvin-final", "sobol056"
identity_key, development_key, _ = spatial.ARCHIVES[stage]
protocol = spatial.load(spatial.STUDY / stage / "protocol.json")
seed_receipts = []
unit = spatial.read_panel(stage, candidate, protocol[development_key], protocol[identity_key], obs, seed_receipts)
grid = [.45] if candidate == "reference" else protocol["density_grid_g_cm3"]
density = spatial.fit_density(obs[fit], unit[fit], grid)
fit_score = spatial.evaluate(spatial.measure(obs[fit], unit[fit], density), scales)
saved = ranked[ranked.stage.eq(stage) & ranked.candidate.eq(candidate)].iloc[0]
assert abs(fit_score["spatial_score"] - saved.spatial_score) < 1e-12
print(f"{candidate} spatial fitting score {fit_score['spatial_score']:.4f}")
print(f"Fitting density {density:.3f} g/cm³, based on {len(seed_receipts)} verified seeds")
print()
print(ranked.loc[ranked.spatial_rank.le(3), ["stage", "candidate", "old_rank", "spatial_rank", "spatial_score"]]
      .round(3).to_string(index=False))

## SO4/T4 joint voxel allocation

The soil-clipped 3D grid preserves the combination of across-row position, along-row position
and depth. A 2D projection sums shares over the omitted horizontal coordinate. Each observed
vine or simulation contributes equally after normalization to spatial shares.

![SO4/T4 allocation by horizontal position and depth](../../Calvin_SO4_Parameter_Exploration_2026-11-18/Spatial_Evaluation_2026-11-19/so4_spatial_depth_maps.png)

In [ ]:
print(so4[["candidate", "seeds", "voxel_cm", "joint_xyz_tv", "across_depth_tv",
           "along_depth_tv", "depth_tv"]].sort_values(["voxel_cm", "joint_xyz_tv"]).round(3).to_string(index=False))

## Optional rebuild

The notebook can regenerate the local tables from the archived receipts and prepared targets.
Calvin uses saved predictions. SO4 runs 288 frozen simulations, compares exact archived run
identities and writes the new voxel summaries and seed records. For the SO4 replay, use the
qualified Python 3.10 engine environment and the project dependency path from the detailed report.
The figures can be regenerated from the resulting tables.

In [ ]:
rebuild_calvin = False
replay_so4 = False
replot = False
for enabled, script in ((rebuild_calvin, "calvin_spatial.py"), (replay_so4, "so4_spatial.py"), (replot, "plot_spatial.py")):
    if enabled:
        subprocess.run([sys.executable, str(directory / script)], cwd=root, check=True)